# Load a 3D NIfTI with itkwidgets (JupyterLite)

This notebook runs in the **Pyodide** kernel. Dependencies come from the docs checked into `PythonLightSuite/docs/`:

- **LightSuite** (`docs/LightSuite/`): atlas pipelines use NIfTI files; the Python helper `scripts/lsfm_atlas_inspect` loads volumes with **nibabel**, **numpy**, **matplotlib**, and **pandas**.
- **ITK-Wasm** (`docs/ITK-Wasm-main/`): WASM-friendly I/O via **itkwasm** / **itkwasm-image-io**. On Pyodide/emscripten only async readers exist — use **`await imread_async(...)`**, not `imread`.
- **itkwidgets** (`docs/itkwidgets-main`): interactive web viewers via `itkwidgets.view`; in JupyterLite use **`piplite`** and install **`imjoy-jupyterlab-extension`** in the site build (see [itkwidgets deployments](https://itkwidgets.readthedocs.io/en/latest/deployments.html)). Recent **`ngff-zarr` (≥0.10)** uses **OME-Zarr ≥0.5**, which requires **Zarr 3** and **`numcodecs>=0.14`**. **`numcodecs` has no PyPI wasm wheel**—Pyodide ships **0.13.x**—so upgrading Zarr with micropip fails. Here we **pin `ngff-zarr==0.9.1`** (still satisfies itkwidgets **≥0.8.7**) and **`zarr>=2,<3`** so `view()` stays on the Zarr **2** stack.

Below we fetch a small public `.nii`, read it with `await imread_async(...)`, and display it with itkwidgets. If you previously installed other versions in this tab, use **Restart kernel** and run from the top so **`reinstall=True`** can replace stale packages.


In [ ]:
import piplite

# Zarr 3 needs numcodecs>=0.14; PyPI has no Emscripten wheels for numcodecs, and Pyodide's
# bundled numcodecs is 0.13.x — so micropip cannot satisfy Zarr 3's deps. Pin ngff-zarr to
# 0.9.x (before 0.10's Zarr-3 / OME-0.5 writer path) + zarr 2.x — compatible with itkwidgets 1.0a55.
# reinstall=True: replace a newer ngff-zarr left over from an earlier session (e.g. 0.34).
await piplite.install(
    [
        "itkwidgets==1.0a55",
        "ngff-zarr[dask-image]==0.9.1",
        "zarr>=2.14,<3",
        "itkwasm-image-io",
    ],
    reinstall=True,
)


In [ ]:
from pyodide.http import pyfetch

# Small test volume from nibabel (BSD); suitable for smoke-testing I/O + viewer
NIFTI_URL = (
    "https://raw.githubusercontent.com/nipy/nibabel/master/"
    "nibabel/tests/data/anatomical.nii"
)


async def download(url: str, path: str) -> int:
    resp = await pyfetch(url)
    data = await resp.bytes()
    with open(path, "wb") as f:
        f.write(data)
    return len(data)


local_path = "anatomical.nii"
nbytes = await download(NIFTI_URL, local_path)
print(f"Downloaded {local_path} ({nbytes} bytes)")


In [ ]:
import warnings

# Upstream itkwasm_image_io_emscripten still uses JsProxy.as_object_map(); Pyodide prefers as_py_json().
warnings.filterwarnings("ignore", message=r".*as_object_map.*", category=RuntimeWarning)

from itkwasm_image_io import imread_async
from itkwidgets import view

image = await imread_async(local_path)
view(image, rotate=True)
